# Relatório Exploratório do Target

Notebook para gerar o relatório HTML do target, com botões para baixar tabelas e imagens, além de exportação automática de tabelas `.tex` e figuras `.pdf`/`.png` para LaTeX.

In [2]:
# ============================================================
# RELATÓRIO EXPLORATÓRIO DO TARGET
# HTML + BOTÕES DE DOWNLOAD + EXPORTAÇÃO PARA LaTeX
#
# Entrada:
# - creditcard.csv
#
# Saídas:
# - relatorio_target.html
# - relatorio_target/
#     ├── tabela_resumo_target.tex
#     ├── tabela_distribuicao_target.tex
#     ├── tabela_matrizes_baseline.tex
#     ├── grafico_frequencia_absoluta.pdf
#     ├── grafico_frequencia_absoluta.png
#     ├── grafico_frequencia_relativa.pdf
#     ├── grafico_frequencia_relativa.png
#     ├── grafico_serie_acumulada_fraudes.pdf
#     ├── grafico_serie_acumulada_fraudes.png
#     ├── matriz_confusao_ideal.pdf
#     ├── matriz_confusao_ideal.png
#     ├── matriz_confusao_tudo_nao_fraude.pdf
#     ├── matriz_confusao_tudo_nao_fraude.png
#     ├── matriz_confusao_tudo_fraude.pdf
#     ├── matriz_confusao_tudo_fraude.png
#     └── comandos_latex_exemplo.tex
# ============================================================

import io
import html
import base64
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix

import plotly.graph_objects as go
import plotly.io as pio


# ============================================================
# FUNÇÕES AUXILIARES DE FORMATAÇÃO
# ============================================================

def formatar_numero(valor, casas=2):
    if pd.isna(valor):
        return "NA"

    if np.isinf(valor):
        return "Infinito"

    return f"{valor:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def formatar_numero_latex(valor, casas=4):
    if pd.isna(valor):
        return "-"

    if np.isinf(valor):
        return r"$\infty$"

    try:
        return f"{float(valor):.{casas}f}"
    except Exception:
        return str(valor)


def formatar_inteiro(valor):
    return f"{int(valor):,}".replace(",", ".")


def formatar_percentual_html(valor, casas=4):
    if pd.isna(valor):
        return "-"

    return f"{float(valor):.{casas}f}%".replace(".", ",")


def formatar_percentual_latex(valor, casas=4):
    if pd.isna(valor):
        return "-"

    return f"{float(valor):.{casas}f}\\%"


def fig_plotly_to_html(fig):
    return pio.to_html(
        fig,
        full_html=False,
        include_plotlyjs=False,
        config={
            "displayModeBar": True,
            "scrollZoom": True,
            "responsive": True,
            "displaylogo": False,
            "toImageButtonOptions": {
                "format": "png",
                "filename": "grafico_target",
                "height": 900,
                "width": 1400,
                "scale": 2
            },
            "modeBarButtonsToRemove": ["lasso2d", "select2d"]
        }
    )


def fig_to_base64(fig):
    buffer = io.BytesIO()

    fig.savefig(
        buffer,
        format="png",
        dpi=150,
        bbox_inches="tight"
    )

    buffer.seek(0)

    return base64.b64encode(buffer.read()).decode("utf-8")


# ============================================================
# CÁLCULOS
# ============================================================

def calcular_entropia_shannon(probabilidades):
    probabilidades = np.asarray(probabilidades, dtype=float)
    probabilidades = probabilidades[probabilidades > 0]

    entropia = -np.sum(probabilidades * np.log2(probabilidades))

    if len(probabilidades) <= 1:
        entropia_normalizada = 0.0
    else:
        entropia_normalizada = entropia / np.log2(len(probabilidades))

    return entropia, entropia_normalizada


def calcular_matriz_confusao(y_true, y_pred):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    )

    tn, fp, fn, tp = cm.ravel()

    return {
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp)
    }


def preparar_valores_matriz(cm):
    tn = cm["tn"]
    fp = cm["fp"]
    fn = cm["fn"]
    tp = cm["tp"]

    total_fraudes = fn + tp
    total_nao_fraudes = tn + fp

    fn_pct = fn / total_fraudes * 100 if total_fraudes > 0 else 0
    tp_pct = tp / total_fraudes * 100 if total_fraudes > 0 else 0

    tn_pct = tn / total_nao_fraudes * 100 if total_nao_fraudes > 0 else 0
    fp_pct = fp / total_nao_fraudes * 100 if total_nao_fraudes > 0 else 0

    return {
        "fn": {
            "pct": fn_pct,
            "count": fn,
            "qualidade": 100 - fn_pct
        },
        "tp": {
            "pct": tp_pct,
            "count": tp,
            "qualidade": tp_pct
        },
        "tn": {
            "pct": tn_pct,
            "count": tn,
            "qualidade": tn_pct
        },
        "fp": {
            "pct": fp_pct,
            "count": fp,
            "qualidade": 100 - fp_pct
        }
    }


def preparar_valores_matriz_ideal(n_nao_fraude, n_fraude):
    return {
        "fn": {
            "pct": 0.0,
            "count": 0,
            "qualidade": 100.0
        },
        "tp": {
            "pct": 100.0,
            "count": n_fraude,
            "qualidade": 100.0
        },
        "tn": {
            "pct": 100.0,
            "count": n_nao_fraude,
            "qualidade": 100.0
        },
        "fp": {
            "pct": 0.0,
            "count": 0,
            "qualidade": 100.0
        }
    }


def cor_por_qualidade(q):
    if q >= 95:
        return "cell q95"
    elif q >= 85:
        return "cell q85"
    elif q >= 70:
        return "cell q70"
    elif q >= 50:
        return "cell q50"
    elif q >= 30:
        return "cell q30"
    else:
        return "cell q10"


# ============================================================
# HTML: MATRIZ DE CONFUSÃO
# ============================================================

def gerar_html_matriz(titulo, valores, matriz_ideal=False, imagem_base64=None, nome_imagem=None):
    if matriz_ideal:
        desc_fn = "Ideal: nenhuma fraude perdida"
        desc_tp = "Ideal: fraudes detectadas"
        desc_tn = "Ideal: não fraudes corretas"
        desc_fp = "Ideal: nenhum falso alerta"
    else:
        desc_fn = "Erro: fraude perdida"
        desc_tp = "Acerto: fraude detectada"
        desc_tn = "Acerto: não fraude"
        desc_fp = "Erro: falso alerta"

    botao_imagem = ""

    if imagem_base64 is not None and nome_imagem is not None:
        botao_imagem = f"""
        <div class="section-actions-only">
            <a
                class="download-btn link-btn"
                href="data:image/png;base64,{imagem_base64}"
                download="{html.escape(nome_imagem)}"
            >
                Baixar PNG
            </a>
        </div>
        """

    html_matriz = f"""
    <section class="matrix-card">
        <div class="section-header">
            <h2>{html.escape(titulo)}</h2>
        </div>

        {botao_imagem}

        <div class="matrix-area">

            <div class="matrix-wrapper">

                <div class="corner"></div>
                <div class="x-label">Pred Não Fraude</div>
                <div class="x-label">Pred Fraude</div>

                <div class="y-label">Real Fraude</div>

                <div class="{cor_por_qualidade(valores['fn']['qualidade'])}">
                    <div class="pct">{valores['fn']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['fn']['count'])})</div>
                    <div class="cell-desc">{desc_fn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['tp']['qualidade'])}">
                    <div class="pct">{valores['tp']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['tp']['count'])})</div>
                    <div class="cell-desc">{desc_tp}</div>
                </div>

                <div class="y-label">Real Não Fraude</div>

                <div class="{cor_por_qualidade(valores['tn']['qualidade'])}">
                    <div class="pct">{valores['tn']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['tn']['count'])})</div>
                    <div class="cell-desc">{desc_tn}</div>
                </div>

                <div class="{cor_por_qualidade(valores['fp']['qualidade'])}">
                    <div class="pct">{valores['fp']['pct']:.2f}%</div>
                    <div class="count">({formatar_inteiro(valores['fp']['count'])})</div>
                    <div class="cell-desc">{desc_fp}</div>
                </div>

            </div>

            <div class="quality-legend">
                <div class="quality-title">Qualidade</div>
                <div class="quality-bar"></div>
                <div class="quality-top">Melhor</div>
                <div class="quality-bottom">Pior</div>
            </div>

        </div>
    </section>
    """

    return html_matriz


# ============================================================
# HTML: TABELAS COM BOTÃO
# ============================================================

def gerar_tabela_html(df, table_id, classe="data-table"):
    if df is None or df.empty:
        return "<p>Nenhum dado disponível.</p>"

    html_tabela = f'<table id="{html.escape(str(table_id))}" class="{classe}">\n'

    html_tabela += "<thead><tr>"

    for col in df.columns:
        html_tabela += f"<th>{html.escape(str(col))}</th>"

    html_tabela += "</tr></thead>\n<tbody>\n"

    for _, row in df.iterrows():
        html_tabela += "<tr>"

        for valor in row:
            html_tabela += f"<td>{html.escape(str(valor))}</td>"

        html_tabela += "</tr>\n"

    html_tabela += "</tbody></table>"

    return html_tabela


def gerar_secao_tabela(titulo, tabela_html, table_id, nome_csv):
    return f"""
    <section class="card">
        <div class="section-header">
            <h2>{html.escape(titulo)}</h2>
            <button
                class="download-btn"
                onclick="baixarTabelaCSV('{html.escape(str(table_id))}', '{html.escape(str(nome_csv))}')"
            >
                Baixar CSV
            </button>
        </div>

        <div class="table-wrapper">
            {tabela_html}
        </div>
    </section>
    """


# ============================================================
# FIGURAS MATPLOTLIB PARA LATEX E DOWNLOAD
# ============================================================

def gerar_fig_frequencia_absoluta(n_nao_fraude, n_fraude):
    fig, ax = plt.subplots(figsize=(8, 5.5))

    classes = ["Não Fraude", "Fraude"]
    valores = [n_nao_fraude, n_fraude]

    barras = ax.bar(classes, valores)

    ax.set_title(
        "Frequência Absoluta das Classes do Target",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Classe", fontweight="bold")
    ax.set_ylabel("Quantidade", fontweight="bold")

    ax.grid(axis="y", alpha=0.25)

    for barra, valor in zip(barras, valores):
        ax.text(
            barra.get_x() + barra.get_width() / 2,
            valor,
            formatar_inteiro(valor),
            ha="center",
            va="bottom",
            fontweight="bold"
        )

    plt.tight_layout()

    return fig


def gerar_fig_frequencia_relativa(percentual_nao_fraude, percentual_fraude):
    fig, ax = plt.subplots(figsize=(9, 5.5))

    classes = ["Não Fraude", "Fraude"]
    valores = [percentual_nao_fraude, percentual_fraude]

    barras = ax.barh(classes, valores)

    ax.set_title(
        "Frequência Relativa das Classes do Target",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Percentual (%)", fontweight="bold")
    ax.set_ylabel("Classe", fontweight="bold")

    ax.grid(axis="x", alpha=0.25)
    ax.invert_yaxis()

    for barra, valor in zip(barras, valores):
        ax.text(
            valor,
            barra.get_y() + barra.get_height() / 2,
            f" {valor:.4f}%",
            va="center",
            fontweight="bold"
        )

    plt.tight_layout()

    return fig


def gerar_fig_serie_acumulada(serie_transacao, serie_fraude_acumulada):
    fig, ax = plt.subplots(figsize=(11, 5.8))

    ax.plot(
        serie_transacao,
        serie_fraude_acumulada,
        linewidth=1.8
    )

    ax.set_title(
        "Série Acumulada de Fraudes por Ordem da Transação",
        fontsize=14,
        fontweight="bold"
    )

    ax.set_xlabel("Índice da transação", fontweight="bold")
    ax.set_ylabel("Quantidade acumulada de fraudes", fontweight="bold")

    ax.grid(alpha=0.25)

    plt.tight_layout()

    return fig


def gerar_fig_matriz_confusao(titulo, valores):
    matriz_pct = np.array([
        [valores["fn"]["pct"], valores["tp"]["pct"]],
        [valores["tn"]["pct"], valores["fp"]["pct"]]
    ])

    matriz_count = np.array([
        [valores["fn"]["count"], valores["tp"]["count"]],
        [valores["tn"]["count"], valores["fp"]["count"]]
    ])

    fig, ax = plt.subplots(figsize=(8.5, 6.2))

    im = ax.imshow(
        matriz_pct,
        vmin=0,
        vmax=100,
        cmap="Blues"
    )

    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label("Percentual por classe real (%)", fontweight="bold")

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])

    ax.set_xticklabels(["Pred Não Fraude", "Pred Fraude"], fontweight="bold")
    ax.set_yticklabels(["Real Fraude", "Real Não Fraude"], fontweight="bold")

    ax.set_title(
        titulo,
        fontsize=14,
        fontweight="bold",
        pad=14
    )

    textos = [
        ["Fraude perdida", "Fraude detectada"],
        ["Não fraude correta", "Falso alerta"]
    ]

    for i in range(2):
        for j in range(2):
            ax.text(
                j,
                i,
                f"{matriz_pct[i, j]:.2f}%\n({formatar_inteiro(matriz_count[i, j])})\n{textos[i][j]}",
                ha="center",
                va="center",
                color="black",
                fontweight="bold",
                fontsize=10
            )

    plt.tight_layout()

    return fig


def salvar_figura(fig, caminho_sem_extensao):
    caminho_sem_extensao = Path(caminho_sem_extensao)

    fig.savefig(
        caminho_sem_extensao.with_suffix(".pdf"),
        bbox_inches="tight"
    )

    fig.savefig(
        caminho_sem_extensao.with_suffix(".png"),
        dpi=300,
        bbox_inches="tight"
    )


# ============================================================
# EXPORTAÇÃO LATEX
# ============================================================

def preparar_df_latex(df):
    df_latex = df.copy()

    for col in df_latex.columns:
        if pd.api.types.is_float_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: formatar_numero_latex(x, 6))
        elif pd.api.types.is_integer_dtype(df_latex[col]):
            df_latex[col] = df_latex[col].apply(lambda x: int(x) if not pd.isna(x) else x)

    return df_latex


def salvar_tabela_latex(df, caminho, caption, label, longtable=False):
    caminho = Path(caminho)

    if df is None or df.empty:
        caminho.write_text(
            "% Tabela vazia: nenhum dado disponível.\n",
            encoding="utf-8"
        )
        return

    df_latex = preparar_df_latex(df)

    tex = df_latex.to_latex(
        index=False,
        escape=True,
        longtable=longtable,
        caption=caption,
        label=label
    )

    caminho.write_text(tex, encoding="utf-8")


def exportar_artefatos_latex(
    pasta_latex,
    tabela_resumo_target,
    tabela_distribuicao_target,
    tabela_matrizes_baseline,
    figs
):
    pasta_latex = Path(pasta_latex)
    pasta_latex.mkdir(parents=True, exist_ok=True)

    salvar_tabela_latex(
        tabela_resumo_target,
        pasta_latex / "tabela_resumo_target.tex",
        caption="Resumo exploratório da variável alvo.",
        label="tab:resumo-target",
        longtable=False
    )

    salvar_tabela_latex(
        tabela_distribuicao_target,
        pasta_latex / "tabela_distribuicao_target.tex",
        caption="Distribuição absoluta e relativa das classes do target.",
        label="tab:distribuicao-target",
        longtable=False
    )

    salvar_tabela_latex(
        tabela_matrizes_baseline,
        pasta_latex / "tabela_matrizes_baseline.tex",
        caption="Matrizes de confusão dos cenários de referência para o target.",
        label="tab:matrizes-baseline-target",
        longtable=False
    )

    for nome, fig in figs.items():
        salvar_figura(
            fig,
            pasta_latex / nome
        )

    comandos_latex = r"""
% ============================================================
% COMANDOS LaTeX DE EXEMPLO
%
% Pacotes recomendados no preâmbulo:
% \usepackage{graphicx}
% \usepackage{float}
% \usepackage{booktabs}
% \usepackage{longtable}
% \usepackage{pdflscape}
% ============================================================

% ----------------------------
% Tabela resumo do target
% ----------------------------
\begin{table}[H]
\centering
\caption{Resumo exploratório da variável alvo.}
\label{tab:resumo-target-main}
\input{relatorio_target/tabela_resumo_target.tex}
\end{table}

% ----------------------------
% Distribuição do target
% ----------------------------
\begin{table}[H]
\centering
\caption{Distribuição absoluta e relativa das classes do target.}
\label{tab:distribuicao-target-main}
\input{relatorio_target/tabela_distribuicao_target.tex}
\end{table}

% ----------------------------
% Baselines triviais
% ----------------------------
\begin{table}[H]
\centering
\caption{Matrizes de confusão dos cenários de referência.}
\label{tab:matrizes-baseline-target-main}
\input{relatorio_target/tabela_matrizes_baseline.tex}
\end{table}

% ----------------------------
% Frequência absoluta
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.75\textwidth]{relatorio_target/grafico_frequencia_absoluta.pdf}
\caption{Frequência absoluta das classes do target.}
\label{fig:target-freq-absoluta}
\end{figure}

% ----------------------------
% Frequência relativa
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.75\textwidth]{relatorio_target/grafico_frequencia_relativa.pdf}
\caption{Frequência relativa das classes do target.}
\label{fig:target-freq-relativa}
\end{figure}

% ----------------------------
% Série acumulada
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=\textwidth]{relatorio_target/grafico_serie_acumulada_fraudes.pdf}
\caption{Série acumulada de fraudes pela ordem das transações.}
\label{fig:target-serie-acumulada}
\end{figure}

% ----------------------------
% Matriz ideal
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.85\textwidth]{relatorio_target/matriz_confusao_ideal.pdf}
\caption{Matriz de confusão do cenário ideal.}
\label{fig:matriz-target-ideal}
\end{figure}

% ----------------------------
% Matriz baseline: tudo não fraude
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.85\textwidth]{relatorio_target/matriz_confusao_tudo_nao_fraude.pdf}
\caption{Matriz de confusão do baseline que classifica todas as transações como não fraude.}
\label{fig:matriz-target-tudo-nao-fraude}
\end{figure}

% ----------------------------
% Matriz baseline: tudo fraude
% ----------------------------
\begin{figure}[H]
\centering
\includegraphics[width=0.85\textwidth]{relatorio_target/matriz_confusao_tudo_fraude.pdf}
\caption{Matriz de confusão do baseline que classifica todas as transações como fraude.}
\label{fig:matriz-target-tudo-fraude}
\end{figure}
"""

    (pasta_latex / "comandos_latex_exemplo.tex").write_text(
        comandos_latex,
        encoding="utf-8"
    )


# ============================================================
# FUNÇÃO PRINCIPAL
# ============================================================

def gerar_relatorio_target(
    arquivo_dados="creditcard.csv",
    nome_target=None,
    pasta_saida=".",
    nome_arquivo_saida="relatorio_target.html",
    pasta_latex="relatorio_target",
    exportar_latex=True
):

    # ========================================================
    # LEITURA DOS DADOS
    # ========================================================

    df = pd.read_csv(arquivo_dados)

    if nome_target is None:
        if "status_fraude" in df.columns:
            nome_target = "status_fraude"

        elif "Class" in df.columns:
            nome_target = "Class"

        else:
            raise ValueError(
                "Não encontrei automaticamente o target. "
                "Informe nome_target='sua_coluna'."
            )

    if nome_target not in df.columns:
        raise ValueError(f"A coluna target '{nome_target}' não existe no arquivo.")

    y = df[nome_target].dropna().astype(int).copy()

    valores_unicos = sorted(y.unique())

    if set(valores_unicos) != {0, 1}:
        raise ValueError(
            f"O target precisa ser binário com valores 0 e 1. "
            f"Valores encontrados: {valores_unicos}"
        )

    # ========================================================
    # ESTATÍSTICAS BÁSICAS DO TARGET
    # ========================================================

    total = len(y)

    n_nao_fraude = int((y == 0).sum())
    n_fraude = int((y == 1).sum())

    p_nao_fraude = n_nao_fraude / total
    p_fraude = n_fraude / total

    percentual_nao_fraude = p_nao_fraude * 100
    percentual_fraude = p_fraude * 100

    if n_nao_fraude >= n_fraude:
        classe_majoritaria = "Não Fraude"
        classe_minoritaria = "Fraude"
        n_majoritaria = n_nao_fraude
        n_minoritaria = n_fraude
    else:
        classe_majoritaria = "Fraude"
        classe_minoritaria = "Não Fraude"
        n_majoritaria = n_fraude
        n_minoritaria = n_nao_fraude

    razao_desbalanceamento = (
        n_majoritaria / n_minoritaria
        if n_minoritaria > 0
        else np.inf
    )

    entropia, entropia_normalizada = calcular_entropia_shannon(
        [p_nao_fraude, p_fraude]
    )

    # ========================================================
    # BASELINES TRIVIAIS
    # ========================================================

    y_baseline_tudo_nao_fraude = np.zeros_like(y)
    y_baseline_tudo_fraude = np.ones_like(y)

    cm_tudo_nao_fraude = calcular_matriz_confusao(
        y_true=y,
        y_pred=y_baseline_tudo_nao_fraude
    )

    cm_tudo_fraude = calcular_matriz_confusao(
        y_true=y,
        y_pred=y_baseline_tudo_fraude
    )

    valores_tudo_nao_fraude = preparar_valores_matriz(
        cm_tudo_nao_fraude
    )

    valores_tudo_fraude = preparar_valores_matriz(
        cm_tudo_fraude
    )

    valores_ideal = preparar_valores_matriz_ideal(
        n_nao_fraude=n_nao_fraude,
        n_fraude=n_fraude
    )

    # ========================================================
    # TABELAS PARA HTML/LaTeX
    # ========================================================

    tabela_resumo_target = pd.DataFrame([
        {
            "Métrica": "Total de observações",
            "Valor": total,
            "Descrição": "Quantidade total de registros válidos no target."
        },
        {
            "Métrica": "Não fraudes",
            "Valor": n_nao_fraude,
            "Descrição": f"{percentual_nao_fraude:.6f}% da base."
        },
        {
            "Métrica": "Fraudes",
            "Valor": n_fraude,
            "Descrição": f"{percentual_fraude:.6f}% da base."
        },
        {
            "Métrica": "Classe majoritária",
            "Valor": classe_majoritaria,
            "Descrição": f"{formatar_inteiro(n_majoritaria)} observações."
        },
        {
            "Métrica": "Classe minoritária",
            "Valor": classe_minoritaria,
            "Descrição": f"{formatar_inteiro(n_minoritaria)} observações."
        },
        {
            "Métrica": "Razão de desbalanceamento",
            "Valor": f"{razao_desbalanceamento:.6f}",
            "Descrição": "Razão entre classe majoritária e classe minoritária."
        },
        {
            "Métrica": "Probabilidade empírica de fraude",
            "Valor": f"{p_fraude:.10f}",
            "Descrição": f"Equivalente a {percentual_fraude:.6f}%."
        },
        {
            "Métrica": "Entropia de Shannon",
            "Valor": f"{entropia:.10f}",
            "Descrição": "Incerteza da distribuição binária do target."
        },
        {
            "Métrica": "Entropia de Shannon normalizada",
            "Valor": f"{entropia_normalizada:.10f}",
            "Descrição": "Entropia dividida pela entropia máxima possível para duas classes."
        },
    ])

    tabela_distribuicao_target = pd.DataFrame([
        {
            "Classe": "Não Fraude",
            "Quantidade": n_nao_fraude,
            "Percentual": percentual_nao_fraude
        },
        {
            "Classe": "Fraude",
            "Quantidade": n_fraude,
            "Percentual": percentual_fraude
        },
    ])

    tabela_distribuicao_target_html = tabela_distribuicao_target.copy()
    tabela_distribuicao_target_html["Quantidade"] = tabela_distribuicao_target_html["Quantidade"].apply(formatar_inteiro)
    tabela_distribuicao_target_html["Percentual"] = tabela_distribuicao_target_html["Percentual"].apply(lambda x: f"{x:.6f}%")

    tabela_matrizes_baseline = pd.DataFrame([
        {
            "Cenário": "Ideal",
            "TN": n_nao_fraude,
            "FP": 0,
            "FN": 0,
            "TP": n_fraude
        },
        {
            "Cenário": "Tudo como Não Fraude",
            "TN": cm_tudo_nao_fraude["tn"],
            "FP": cm_tudo_nao_fraude["fp"],
            "FN": cm_tudo_nao_fraude["fn"],
            "TP": cm_tudo_nao_fraude["tp"]
        },
        {
            "Cenário": "Tudo como Fraude",
            "TN": cm_tudo_fraude["tn"],
            "FP": cm_tudo_fraude["fp"],
            "FN": cm_tudo_fraude["fn"],
            "TP": cm_tudo_fraude["tp"]
        },
    ])

    # ========================================================
    # SÉRIE ACUMULADA DE FRAUDES
    # ========================================================

    serie_transacao = np.arange(1, total + 1)
    serie_fraude_acumulada = y.cumsum().to_numpy()

    # ========================================================
    # GRÁFICOS PLOTLY PARA HTML
    # ========================================================

    fig_abs = go.Figure()

    fig_abs.add_trace(
        go.Bar(
            x=["Não Fraude", "Fraude"],
            y=[n_nao_fraude, n_fraude],
            text=[
                formatar_inteiro(n_nao_fraude),
                formatar_inteiro(n_fraude)
            ],
            textposition="outside",
            marker=dict(
                color=["#2563eb", "#facc15"],
                line=dict(
                    color="#111827",
                    width=1
                )
            ),
            hovertemplate=(
                "Classe: %{x}<br>"
                "Quantidade: %{y:,}"
                "<extra></extra>"
            )
        )
    )

    fig_abs.update_layout(
        height=520,
        margin=dict(l=50, r=30, t=60, b=60),
        title=dict(
            text="Frequência Absoluta das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Classe",
        yaxis_title="Quantidade",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_abs.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    grafico_abs_html = fig_plotly_to_html(fig_abs)

    fig_rel = go.Figure()

    fig_rel.add_trace(
        go.Bar(
            y=["Não Fraude", "Fraude"],
            x=[percentual_nao_fraude, percentual_fraude],
            orientation="h",
            text=[
                f"{percentual_nao_fraude:.4f}%",
                f"{percentual_fraude:.4f}%"
            ],
            textposition="outside",
            marker=dict(
                color=["#2563eb", "#facc15"],
                line=dict(
                    color="#111827",
                    width=1
                )
            ),
            hovertemplate=(
                "Classe: %{y}<br>"
                "Percentual: %{x:.6f}%"
                "<extra></extra>"
            )
        )
    )

    fig_rel.update_layout(
        height=520,
        margin=dict(l=110, r=80, t=60, b=60),
        title=dict(
            text="Frequência Relativa das Classes do Target",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Percentual (%)",
        yaxis_title="Classe",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_rel.update_xaxes(
        range=[0, max(105, percentual_nao_fraude * 1.12)],
        showgrid=True,
        gridcolor="rgba(148,163,184,0.30)"
    )

    fig_rel.update_yaxes(
        autorange="reversed"
    )

    grafico_rel_html = fig_plotly_to_html(fig_rel)

    fig_serie = go.Figure()

    fig_serie.add_trace(
        go.Scattergl(
            x=serie_transacao,
            y=serie_fraude_acumulada,
            mode="lines",
            name="Fraudes acumuladas",
            line=dict(
                width=2,
                color="#dc2626"
            ),
            hovertemplate=(
                "Transação: %{x}<br>"
                "Fraudes acumuladas: %{y}"
                "<extra></extra>"
            )
        )
    )

    fig_serie.update_layout(
        height=560,
        margin=dict(l=60, r=30, t=70, b=60),
        title=dict(
            text="Série Acumulada de Fraudes por Ordem da Transação",
            x=0.5,
            xanchor="center"
        ),
        xaxis_title="Índice da transação",
        yaxis_title="Quantidade acumulada de fraudes",
        plot_bgcolor="white",
        paper_bgcolor="white"
    )

    fig_serie.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    fig_serie.update_yaxes(
        showgrid=True,
        gridcolor="rgba(148,163,184,0.25)"
    )

    grafico_serie_html = fig_plotly_to_html(fig_serie)

    # ========================================================
    # FIGURAS MATPLOTLIB PARA LATEX E DOWNLOAD
    # ========================================================

    fig_abs_static = gerar_fig_frequencia_absoluta(
        n_nao_fraude=n_nao_fraude,
        n_fraude=n_fraude
    )

    fig_rel_static = gerar_fig_frequencia_relativa(
        percentual_nao_fraude=percentual_nao_fraude,
        percentual_fraude=percentual_fraude
    )

    fig_serie_static = gerar_fig_serie_acumulada(
        serie_transacao=serie_transacao,
        serie_fraude_acumulada=serie_fraude_acumulada
    )

    fig_matriz_ideal = gerar_fig_matriz_confusao(
        "Matriz de Confusão - Cenário Ideal",
        valores_ideal
    )

    fig_matriz_tudo_nao_fraude = gerar_fig_matriz_confusao(
        "Matriz de Confusão - Tudo como Não Fraude",
        valores_tudo_nao_fraude
    )

    fig_matriz_tudo_fraude = gerar_fig_matriz_confusao(
        "Matriz de Confusão - Tudo como Fraude",
        valores_tudo_fraude
    )

    imagens_base64 = {
        "grafico_frequencia_absoluta": fig_to_base64(fig_abs_static),
        "grafico_frequencia_relativa": fig_to_base64(fig_rel_static),
        "grafico_serie_acumulada_fraudes": fig_to_base64(fig_serie_static),
        "matriz_confusao_ideal": fig_to_base64(fig_matriz_ideal),
        "matriz_confusao_tudo_nao_fraude": fig_to_base64(fig_matriz_tudo_nao_fraude),
        "matriz_confusao_tudo_fraude": fig_to_base64(fig_matriz_tudo_fraude),
    }

    # ========================================================
    # EXPORTAR LATEX
    # ========================================================

    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)
    caminho_latex = pasta_saida / pasta_latex

    if exportar_latex:
        exportar_artefatos_latex(
            pasta_latex=caminho_latex,
            tabela_resumo_target=tabela_resumo_target,
            tabela_distribuicao_target=tabela_distribuicao_target,
            tabela_matrizes_baseline=tabela_matrizes_baseline,
            figs={
                "grafico_frequencia_absoluta": fig_abs_static,
                "grafico_frequencia_relativa": fig_rel_static,
                "grafico_serie_acumulada_fraudes": fig_serie_static,
                "matriz_confusao_ideal": fig_matriz_ideal,
                "matriz_confusao_tudo_nao_fraude": fig_matriz_tudo_nao_fraude,
                "matriz_confusao_tudo_fraude": fig_matriz_tudo_fraude,
            }
        )

    for fig in [
        fig_abs_static,
        fig_rel_static,
        fig_serie_static,
        fig_matriz_ideal,
        fig_matriz_tudo_nao_fraude,
        fig_matriz_tudo_fraude
    ]:
        plt.close(fig)

    # ========================================================
    # HTML DAS MATRIZES
    # ========================================================

    html_matriz_tudo_nao_fraude = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Baseline Trivial: Tudo como Não Fraude",
        valores=valores_tudo_nao_fraude,
        imagem_base64=imagens_base64["matriz_confusao_tudo_nao_fraude"],
        nome_imagem="matriz_confusao_tudo_nao_fraude.png"
    )

    html_matriz_tudo_fraude = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Baseline Trivial: Tudo como Fraude",
        valores=valores_tudo_fraude,
        imagem_base64=imagens_base64["matriz_confusao_tudo_fraude"],
        nome_imagem="matriz_confusao_tudo_fraude.png"
    )

    html_matriz_ideal = gerar_html_matriz(
        titulo="Matriz de Confusão (%) - Cenário Ideal",
        valores=valores_ideal,
        matriz_ideal=True,
        imagem_base64=imagens_base64["matriz_confusao_ideal"],
        nome_imagem="matriz_confusao_ideal.png"
    )

    # ========================================================
    # TABELAS HTML
    # ========================================================

    tabela_resumo_target_html = tabela_resumo_target.copy()
    tabela_resumo_target_html["Valor"] = tabela_resumo_target_html["Valor"].astype(str)

    tabela_matrizes_baseline_html = tabela_matrizes_baseline.copy()

    html_tabela_resumo_target = gerar_tabela_html(
        tabela_resumo_target_html,
        table_id="tabela_resumo_target"
    )

    html_tabela_distribuicao_target = gerar_tabela_html(
        tabela_distribuicao_target_html,
        table_id="tabela_distribuicao_target"
    )

    html_tabela_matrizes_baseline = gerar_tabela_html(
        tabela_matrizes_baseline_html,
        table_id="tabela_matrizes_baseline"
    )

    # ========================================================
    # HTML FINAL
    # ========================================================

    html_final = f"""
    <!DOCTYPE html>
    <html lang="pt-BR">
    <head>
        <meta charset="UTF-8">

        <title>Relatório do Target - {nome_target}</title>

        <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

        <style>
            body {{
                margin: 0;
                padding: 32px;
                font-family: Arial, Helvetica, sans-serif;
                background: #f4f6f8;
                color: #020617;
            }}

            .container {{
                max-width: 1450px;
                margin: 0 auto;
            }}

            h1 {{
                text-align: center;
                color: #020617;
                margin-bottom: 30px;
            }}

            h2 {{
                text-align: center;
                color: #020617;
                margin-top: 0;
                margin-bottom: 22px;
                font-size: 22px;
            }}

            .section-header {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 14px;
                flex-wrap: wrap;
                margin-bottom: 18px;
            }}

            .section-header h2 {{
                margin: 0;
            }}

            .section-actions-only {{
                display: flex;
                justify-content: flex-end;
                margin-bottom: 12px;
            }}

            .download-btn {{
                border: 1px solid #bfdbfe;
                background: #eff6ff;
                color: #1e3a8a;
                padding: 8px 12px;
                border-radius: 10px;
                font-weight: 800;
                font-size: 13px;
                cursor: pointer;
                text-decoration: none;
                display: inline-block;
            }}

            .download-btn:hover {{
                background: #dbeafe;
            }}

            .card,
            .matrix-card {{
                background: #ffffff;
                border-radius: 16px;
                padding: 24px;
                margin-bottom: 32px;
                box-shadow: 0 8px 24px rgba(15, 23, 42, 0.08);
            }}

            .grid {{
                display: grid;
                grid-template-columns: repeat(4, 1fr);
                gap: 14px;
            }}

            .metric-card {{
                background: #f8fafc;
                border: 1px solid #e2e8f0;
                border-radius: 14px;
                padding: 16px;
            }}

            .metric-label {{
                font-size: 13px;
                font-weight: 700;
                color: #475569;
                margin-bottom: 8px;
            }}

            .metric-value {{
                font-size: 22px;
                font-weight: 900;
                color: #020617;
                font-family: Consolas, Monaco, monospace;
                word-break: break-word;
            }}

            .metric-sub {{
                margin-top: 6px;
                font-size: 12px;
                color: #64748b;
                line-height: 1.35;
            }}

            .plotly-graph-div {{
                width: 100% !important;
            }}

            .interpretacao {{
                margin-top: 18px;
                background: #f8fafc;
                border-left: 5px solid #2563eb;
                padding: 14px 16px;
                border-radius: 10px;
                line-height: 1.55;
                color: #334155;
            }}

            .alerta {{
                background: #fff7ed;
                border-left: 5px solid #f97316;
            }}

            .table-wrapper {{
                overflow-x: auto;
                border: 1px solid #e2e8f0;
                border-radius: 12px;
                max-height: 580px;
                overflow-y: auto;
            }}

            .data-table {{
                width: 100%;
                border-collapse: collapse;
                font-size: 13px;
                margin-top: 0;
            }}

            .data-table th {{
                background: #0f172a;
                color: white;
                padding: 10px 8px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 1;
            }}

            .data-table td {{
                border-bottom: 1px solid #e2e8f0;
                padding: 8px;
                color: #020617;
                white-space: nowrap;
            }}

            .data-table tr:nth-child(even) {{
                background: #f8fafc;
            }}

            .matrix-area {{
                display: flex;
                align-items: center;
                justify-content: center;
                gap: 34px;
            }}

            .matrix-wrapper {{
                display: grid;
                grid-template-columns: 180px 1fr 1fr;
                grid-template-rows: 48px 190px 190px;
                width: 950px;
            }}

            .corner {{
                background: transparent;
            }}

            .x-label {{
                display: flex;
                align-items: center;
                justify-content: center;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
                border-bottom: 1px solid #e5e7eb;
            }}

            .y-label {{
                display: flex;
                align-items: center;
                justify-content: flex-end;
                padding-right: 18px;
                font-size: 19px;
                font-weight: 700;
                color: #020617;
            }}

            .cell {{
                display: flex;
                flex-direction: column;
                align-items: center;
                justify-content: center;
                min-height: 180px;
                border: 1px solid #e5e7eb;
                font-size: 20px;
                text-align: center;
                color: #020617 !important;
            }}

            .pct {{
                font-size: 30px;
                font-weight: 900;
                margin-bottom: 4px;
                color: #020617 !important;
            }}

            .count {{
                font-size: 24px;
                font-weight: 900;
                margin-bottom: 8px;
                color: #020617 !important;
            }}

            .cell-desc {{
                font-size: 13px;
                font-weight: 700;
                opacity: 1;
                color: #020617 !important;
            }}

            .q95 {{
                background: #08306b;
            }}

            .q85 {{
                background: #08519c;
            }}

            .q70 {{
                background: #2171b5;
            }}

            .q50 {{
                background: #6baed6;
            }}

            .q30 {{
                background: #c6dbef;
            }}

            .q10 {{
                background: #eff6ff;
            }}

            .quality-legend {{
                position: relative;
                display: flex;
                flex-direction: column;
                align-items: center;
                min-width: 115px;
            }}

            .quality-title {{
                font-weight: 800;
                font-size: 15px;
                margin-bottom: 10px;
                color: #020617;
            }}

            .quality-bar {{
                width: 30px;
                height: 310px;
                border-radius: 16px;
                background: linear-gradient(
                    to bottom,
                    #08306b 0%,
                    #08519c 18%,
                    #2171b5 36%,
                    #6baed6 58%,
                    #c6dbef 78%,
                    #eff6ff 100%
                );
                border: 1px solid #cbd5e1;
            }}

            .quality-top {{
                position: absolute;
                top: 43px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            .quality-bottom {{
                position: absolute;
                top: 335px;
                left: 78px;
                font-size: 13px;
                font-weight: 800;
                color: #020617;
            }}

            @media (max-width: 1100px) {{
                .grid {{
                    grid-template-columns: repeat(2, 1fr);
                }}

                .matrix-area {{
                    flex-direction: column;
                }}

                .matrix-wrapper {{
                    width: 100%;
                    grid-template-columns: 150px 1fr 1fr;
                }}
            }}

            @media (max-width: 700px) {{
                .grid {{
                    grid-template-columns: 1fr;
                }}

                body {{
                    padding: 16px;
                }}

                .matrix-wrapper {{
                    grid-template-columns: 120px 1fr 1fr;
                    grid-template-rows: 48px 160px 160px;
                }}

                .pct {{
                    font-size: 22px;
                }}

                .count {{
                    font-size: 18px;
                }}

                .y-label,
                .x-label {{
                    font-size: 14px;
                }}
            }}
        </style>
    </head>

    <body>
        <div class="container">

            <h1>Relatório Exploratório do Target - {nome_target}</h1>

            <section class="card">
                <h2>Resumo Geral do Target</h2>

                <div class="grid">

                    <div class="metric-card">
                        <div class="metric-label">Total de observações</div>
                        <div class="metric-value">{formatar_inteiro(total)}</div>
                        <div class="metric-sub">Quantidade total de registros válidos no target.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Não fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_nao_fraude)}</div>
                        <div class="metric-sub">{percentual_nao_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Fraudes</div>
                        <div class="metric-value">{formatar_inteiro(n_fraude)}</div>
                        <div class="metric-sub">{percentual_fraude:.6f}% da base.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Classe majoritária</div>
                        <div class="metric-value">{classe_majoritaria}</div>
                        <div class="metric-sub">{formatar_inteiro(n_majoritaria)} observações.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Classe minoritária</div>
                        <div class="metric-value">{classe_minoritaria}</div>
                        <div class="metric-sub">{formatar_inteiro(n_minoritaria)} observações.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Razão de desbalanceamento</div>
                        <div class="metric-value">{formatar_numero(razao_desbalanceamento, 2)} : 1</div>
                        <div class="metric-sub">Razão entre classe majoritária e classe minoritária.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Probabilidade empírica de fraude</div>
                        <div class="metric-value">{p_fraude:.8f}</div>
                        <div class="metric-sub">Equivalente a {percentual_fraude:.6f}%.</div>
                    </div>

                    <div class="metric-card">
                        <div class="metric-label">Entropia de Shannon</div>
                        <div class="metric-value">{entropia:.8f}</div>
                        <div class="metric-sub">Incerteza da distribuição binária do target.</div>
                    </div>

                </div>

                <p class="interpretacao alerta">
                    A variável target apresenta forte desbalanceamento entre as classes.
                    A classe minoritária é muito menos frequente, o que torna inadequado avaliar modelos apenas por acurácia.
                    A entropia de Shannon também reforça a concentração da distribuição em uma única classe.
                </p>
            </section>

            {gerar_secao_tabela(
                "Tabela Resumo do Target",
                html_tabela_resumo_target,
                "tabela_resumo_target",
                "tabela_resumo_target.csv"
            )}

            {gerar_secao_tabela(
                "Tabela de Distribuição do Target",
                html_tabela_distribuicao_target,
                "tabela_distribuicao_target",
                "tabela_distribuicao_target.csv"
            )}

            {gerar_secao_tabela(
                "Tabela das Matrizes de Baseline",
                html_tabela_matrizes_baseline,
                "tabela_matrizes_baseline",
                "tabela_matrizes_baseline.csv"
            )}

            <section class="card">
                <div class="section-actions-only">
                    <a
                        class="download-btn link-btn"
                        href="data:image/png;base64,{imagens_base64['grafico_frequencia_absoluta']}"
                        download="grafico_frequencia_absoluta.png"
                    >
                        Baixar PNG
                    </a>
                </div>
                {grafico_abs_html}
            </section>

            <section class="card">
                <div class="section-actions-only">
                    <a
                        class="download-btn link-btn"
                        href="data:image/png;base64,{imagens_base64['grafico_frequencia_relativa']}"
                        download="grafico_frequencia_relativa.png"
                    >
                        Baixar PNG
                    </a>
                </div>
                {grafico_rel_html}
            </section>

            {html_matriz_ideal}

            {html_matriz_tudo_nao_fraude}

            {html_matriz_tudo_fraude}

            <section class="card">
                <div class="section-actions-only">
                    <a
                        class="download-btn link-btn"
                        href="data:image/png;base64,{imagens_base64['grafico_serie_acumulada_fraudes']}"
                        download="grafico_serie_acumulada_fraudes.png"
                    >
                        Baixar PNG
                    </a>
                </div>

                {grafico_serie_html}

                <p class="interpretacao">
                    A série acumulada soma +1 sempre que uma transação é fraude e soma 0 quando é não fraude.
                    Assim, o eixo Y representa o total acumulado de fraudes até cada posição da base.
                    Trechos mais inclinados indicam regiões da ordenação em que fraudes aparecem com maior frequência.
                </p>
            </section>

        </div>

        <script>
            function limparTextoCSV(texto) {{
                if (texto === null || texto === undefined) {{
                    return "";
                }}

                texto = String(texto).replace(/\\n/g, " ").replace(/\\s+/g, " ").trim();

                if (texto.includes(";") || texto.includes('"')) {{
                    texto = '"' + texto.replace(/"/g, '""') + '"';
                }}

                return texto;
            }}

            function baixarTabelaCSV(tableId, filename) {{
                const tabela = document.getElementById(tableId);

                if (!tabela) {{
                    alert("Tabela não encontrada: " + tableId);
                    return;
                }}

                const linhas = [];

                tabela.querySelectorAll("tr").forEach(function(row) {{
                    const celulas = Array.from(row.querySelectorAll("th, td"));
                    const linha = celulas.map(celula => limparTextoCSV(celula.innerText)).join(";");
                    linhas.push(linha);
                }});

                const csv = "\\ufeff" + linhas.join("\\n");
                const blob = new Blob([csv], {{ type: "text/csv;charset=utf-8;" }});
                const url = URL.createObjectURL(blob);

                const link = document.createElement("a");
                link.href = url;
                link.download = filename;
                document.body.appendChild(link);
                link.click();
                document.body.removeChild(link);

                URL.revokeObjectURL(url);
            }}
        </script>
    </body>
    </html>
    """

    # ========================================================
    # SALVAR HTML
    # ========================================================

    caminho_saida = pasta_saida / nome_arquivo_saida

    caminho_saida.write_text(
        html_final,
        encoding="utf-8"
    )

    print("=" * 80)
    print("RELATÓRIO DO TARGET GERADO COM SUCESSO")
    print("=" * 80)
    print(f"Relatório HTML salvo em: {caminho_saida.resolve()}")

    if exportar_latex:
        print(f"Arquivos LaTeX salvos em: {caminho_latex.resolve()}")

    print("=" * 80)

    return {
        "caminho_html": caminho_saida,
        "caminho_latex": caminho_latex,
        "tabela_resumo_target": tabela_resumo_target,
        "tabela_distribuicao_target": tabela_distribuicao_target,
        "tabela_matrizes_baseline": tabela_matrizes_baseline
    }


# ============================================================
# CHAMADA
# ============================================================

resultado_relatorio_target = gerar_relatorio_target(
    arquivo_dados="creditcard.csv",
    nome_target=None,
    pasta_saida=".",
    nome_arquivo_saida="relatorio_target.html",
    pasta_latex="relatorio_target",
    exportar_latex=True
)

resultado_relatorio_target["caminho_html"]


RELATÓRIO DO TARGET GERADO COM SUCESSO
Relatório HTML salvo em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\relatorio_target.html
Arquivos LaTeX salvos em: C:\Users\Vitor Craveiro\Desktop\UNESP\MATÉRIAS FACULDADES\MATÉRIAS 12 SEMESTRES\TRABALHO DE CONCLUSAO DE CURSO 2\relatorio_target


WindowsPath('relatorio_target.html')